In [1]:
import scanpy as sc
import functools
import os
import sys
import traceback
from typing import Dict, Literal, Optional, Tuple
import cellflow
import scanpy as sc
import numpy as np
import functools
from ott.solvers import utils as solver_utils
import optax
from omegaconf import OmegaConf
from typing import NamedTuple, Any
import hydra
import wandb
import anndata as ad
import pandas as pd
import rapids_singlecell as rsc
import anndata as ad
import os
from cellflow.preprocessing import transfer_labels, compute_wknn, centered_pca, project_pca
from cellflow.metrics._metrics import compute_r_squared, compute_e_distance, compute_scalar_mmd, compute_sinkhorn_div


/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [6]:


def compute_metrics(adata_ref: ad.AnnData, adata_ref_for_ct_error: ad.AnnData, adata_pred: ad.AnnData, adata_ood_true: ad.AnnData, n_neighbors: int=1, cell_type_col: str = "cell_type_broad", min_cells_for_dist_metrics: int = 50) -> dict:
    dict_to_log = {}
    ct_true = adata_ood_true.obs[cell_type_col].value_counts().to_frame()
    ct_true = ct_true / ct_true.sum()
    
    compute_wknn(ref_adata=adata_ref_for_ct_error, query_adata=adata_pred, n_neighbors=n_neighbors, ref_rep_key="X_aligned", query_rep_key="X_aligned")
    transfer_labels(query_adata=adata_pred, ref_adata=adata_ref_for_ct_error, label_key=cell_type_col)
    ct_transferred_pred = adata_pred.obs[f"{cell_type_col}_transfer"].value_counts().to_frame()
    ct_transferred_pred/=ct_transferred_pred.sum()
    all_cell_types = list(set(ct_true.index).union(set(ct_transferred_pred.index)))
    df_all_cell_types = pd.DataFrame(index=all_cell_types, data=np.zeros((len(all_cell_types), 2)), columns=["true", "pred"])
    df_all_cell_types["true"] = ct_true
    df_all_cell_types["pred"] = ct_transferred_pred
    df_all_cell_types = df_all_cell_types.fillna(0.0)
    cell_type_fraction_error = np.abs(df_all_cell_types["true"] - df_all_cell_types["pred"]).sum()
    
    all_cell_types_true = list(adata_ood_true.obs[cell_type_col].value_counts()[adata_ood_true.obs[cell_type_col].value_counts()>min_cells_for_dist_metrics].index)
    compute_wknn(ref_adata=adata_ref, query_adata=adata_pred, n_neighbors=n_neighbors, ref_rep_key="X_aligned", query_rep_key="X_aligned")
    transfer_labels(query_adata=adata_pred, ref_adata=adata_ref, label_key=cell_type_col)
    
    e_distance = {}
    r_sq = {}
    mmd = {}
    sdiv_10 = {}
    sdiv_100 = {}
    n_cell_types_covered = 0
    for cell_type in all_cell_types_true: 
        dist_true = adata_ood_true[adata_ood_true.obs[f"{cell_type_col}"]==cell_type].obsm["X_aligned"]
        dist_pred = adata_pred[adata_pred.obs[f"{cell_type_col}_transfer"]==cell_type].obsm["X_aligned"]
        if len(dist_pred) == 0:
            continue
        n_cell_types_covered+=1
        r_sq[f"r_squared_{cell_type}"] = compute_r_squared(dist_true, dist_pred)
        e_distance[f"e_distance_{cell_type}"] = compute_e_distance(dist_true, dist_pred)
        mmd[f"mmd_{cell_type}"] = compute_scalar_mmd(dist_true, dist_pred)
        sdiv_10[f"div_10_{cell_type}"] = np.nan
        sdiv_100[f"div_100_{cell_type}"] = np.nan

    fraction_cell_types_covered = n_cell_types_covered/len(all_cell_types_true)


    # standard metrics
    ood_r_squared = compute_r_squared(adata_ood_true.obsm["X_aligned"], adata_pred.obsm["X_aligned"])
    ood_e_distance = compute_e_distance(adata_ood_true.obsm["X_aligned"], adata_pred.obsm["X_aligned"])
    ood_mmd = compute_scalar_mmd(adata_ood_true.obsm["X_aligned"], adata_pred.obsm["X_aligned"])
    ood_sdiv_10 = np.nan
    ood_sdiv_100 = np.nan

    # metrics to return
    dict_to_log["fraction_cell_types_covered"] = fraction_cell_types_covered
    dict_to_log["cell_type_fraction_error"] = cell_type_fraction_error
    dict_to_log["mean_r_sq_per_cell_type"] = np.mean(list(r_sq.values()))
    dict_to_log["mean_e_distance_per_cell_type"] = np.mean(list(e_distance.values()))
    dict_to_log["mean_mmd_per_cell_type"] = np.mean(list(mmd.values()))
    dict_to_log["mean_sdiv_10_per_cell_type"] = np.mean(list(sdiv_10.values()))
    dict_to_log["mean_sdiv_100_per_cell_type"] = np.mean(list(sdiv_100.values()))
    dict_to_log["median_r_sq_per_cell_type"] = np.median(list(r_sq.values()))
    dict_to_log["median_e_distance_per_cell_type"] = np.median(list(e_distance.values()))
    dict_to_log["median_mmd_per_cell_type"] = np.median(list(mmd.values()))
    dict_to_log["median_sdiv_10_per_cell_type"] = np.median(list(sdiv_10.values()))
    dict_to_log["median_sdiv_100_per_cell_type"] = np.median(list(sdiv_100.values()))
    dict_to_log.update(r_sq)
    dict_to_log.update(e_distance)
    dict_to_log.update(mmd)
    dict_to_log.update(sdiv_10)
    dict_to_log.update(sdiv_100)
    dict_to_log["ood_r_squared"] = ood_r_squared
    dict_to_log["ood_e_distance"] = ood_e_distance
    dict_to_log["ood_mmd"] = ood_mmd
    dict_to_log["ood_sdiv_10"] = ood_sdiv_10
    dict_to_log["ood_sdiv_100"] = ood_sdiv_100
    return dict_to_log

def get_train_embeddings(adata_same_timepoint: ad.AnnData, embeddings: dict[str, np.ndarray]) -> dict[str, Any]:
    conds = adata_same_timepoint.obs.drop_duplicates(subset="condition")
    emb_vectors = {}
    for _,row in conds.iterrows():
        g_1 = row["gene_target_1"]
        g_2  = row["gene_target_2"]
        if g_1=="control" and g_2=="control":
            continue
        elif g_1 != "control" and g_2!= "control":
            emb_vectors[(g_1, g_2)] = (embeddings[g_1] + embeddings[g_2])/2.0
        elif g_1 != "control" and g_2=="control":
            emb_vectors[(g_1, g_2)] = embeddings[g_1]
        elif g_1=="control" and g_2!="control":
            emb_vectors[(g_1, g_2)] = embeddings[g_2]
    return emb_vectors

def find_closest_embedding(emb_0: np.ndarray, reference_embeddings: dict[str, np.ndarray]) -> Tuple[str, ...]:
    closest_emb = None
    closest_dist = np.inf
    for ref, ref_emb in reference_embeddings.items():
        dist = np.sum((emb_0-ref_emb)**2)
        if dist < closest_dist:
            closest_dist = dist
            closest_emb = ref
    return closest_emb



    


In [18]:
adata = sc.read_h5ad("/lustre/groups/ml01/workspace/ot_perturbation/data/zebrafish_new/zebrafish_processed.h5ad")
out_dir = "/lustre/groups/ml01/workspace/ot_perturbation/models/identity/zebrafish/single_condition_closest_embedding"
ood_conds = list(adata[adata.obs["gene_target"]!="control_control"].obs["condition"].unique())
ood_cond_results = {}
embeddings = adata.uns["gene_embeddings"]


for ood_cond in ood_conds:
    print(ood_cond)
    adata_ood_true = adata[adata.obs["condition"]==ood_cond]
    adata_ref_for_ct_error = adata[adata.obs["condition"]!=ood_cond]
    tp = int(ood_cond.split("_")[-1])
    gene_knockout = "_".join(ood_cond.split("_")[:-1])
    row = adata[adata.obs["condition"]==ood_cond].obs.drop_duplicates(subset="condition").iloc[0]
    adata_same_timepoint = adata[adata.obs["timepoint"]==tp]
    same_timepoint_embeddings = get_train_embeddings(adata_same_timepoint, embeddings)
    
    if row["condition"] == "control_control":
        continue
    elif row["gene_target_2"] == "control":
        emb_0 = ood_cond.split("_")[0]
        closest_emb = find_closest_embedding(embeddings[emb_0], same_timepoint_embeddings)
    else:
        emb_0 = (embeddings[ood_cond.split("_")[0]] + embeddings[ood_cond.split("_")[1]])/2.0
        closest_emb = find_closest_embedding(emb_0, same_timepoint_embeddings)
    
    adata_ood_pred = adata[adata.obs["gene_target"]==f"{closest_emb[0]}_{closest_emb[1]}"]
    
    if adata_ood_pred.n_obs > 30000:
        sc.pp.subsample(adata_ood_pred, n_obs=30000)
    if adata_ood_true.n_obs > 30000:
        sc.pp.subsample(adata_ood_true, n_obs=30000)

    ood_cond_results[ood_cond] = compute_metrics(adata_ref=adata, adata_ref_for_ct_error=adata_ref_for_ct_error, adata_pred=adata_ood_pred, adata_ood_true=adata_ood_true)

    #pd.DataFrame.from_dict(ood_cond_results[ood_cond], columns=[ood_cond], orient="index").to_csv(os.path.join(out_dir, f"{ood_cond}_closest_embedding.csv"))
    #break

zc4h2_control_24


RuntimeError: exception occurred! file=/__w/cuvs/cuvs/cpp/src/neighbors/./detail/./fused_l2_knn.cuh line=957: l2Knn: n_query_rows must be > 0
Obtained 46 stack frames
#1 in /home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/libcuvs/lib64/libcuvs.so: void cuvs::neighbors::detail::fusedL2Knn<long, float, false, float>(unsigned long, long*, float*, float const*, float const*, unsigned long, unsigned long, int, bool, bool, CUstream_st*, cuvsDistanceType, float const*, float const*) +0x570 [0x7f23da3ad3f0]
#2 in /home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/libcuvs/lib64/libcuvs.so: void cuvs::neighbors::detail::brute_force_knn_impl<long, long, float, float>(raft::resources const&, std::vector<float*, std::allocator<float*> >&, std::vector<long, std::allocator<long> >&, long, float*, long, long*, float*, long, bool, bool, std::vector<long, std::allocator<long> >*, cuvsDistanceType, float, std::vector<float*, std::allocator<float*> >

In [21]:
adata_ood_pred = adata[adata.obs["gene_target"]==f"{closest_emb[0]}_{closest_emb[1]}"]
adata_ood_pred.n_obs

73264

In [22]:
closest_emb

('zc4h2', 'control')

In [20]:
adata.obs["gene_target"]

A03_B01_P01-A01_LIG104    control_control
A03_B01_P01-A02_LIG101    control_control
A03_B01_P01-A02_LIG127    control_control
A03_B01_P01-A02_LIG257    control_control
A03_B01_P01-A03_LIG201    control_control
                               ...       
H07_D12_P04-H12_LIG258      mafba_control
H07_D12_P04-H12_LIG264     tfap2a_control
H07_D12_P04-H12_LIG302      mafba_control
H07_D12_P04-H12_LIG370      foxd3_control
H07_D12_P04-H12_LIG6       tfap2a_control
Name: gene_target, Length: 2686684, dtype: category
Categories (24, object): ['cdx4_cdx1a', 'cdx4_control', 'control_control', 'egr2b_control', ..., 'tfap2a_control', 'tfap2a_foxd3', 'wnt3a_wnt8', 'zc4h2_control']

In [19]:
adata.n_obs, adata_ref_for_ct_error.n_obs, adata_ood_pred.n_obs

(2686684, 2663171, 0)

In [14]:
emb_0

['zc4h2', 'control']

In [13]:
embeddings

{'cdx1a': array([ 0.04072264, -0.00771822,  0.00958598, ..., -0.00357641,
        -0.17223947,  0.07910734], dtype=float32),
 'cdx4': array([ 0.04544561,  0.0085295 ,  0.02384006, ..., -0.00804145,
        -0.16306993,  0.0667288 ], dtype=float32),
 'control': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'egr2b': array([ 0.03005039,  0.02441801,  0.01638456, ..., -0.00273611,
        -0.1044653 ,  0.06679769], dtype=float32),
 'epha4a': array([ 0.01345224,  0.00413656, -0.04197775, ...,  0.01749663,
        -0.10178802, -0.02238981], dtype=float32),
 'foxd3': array([ 0.03822063,  0.01314073,  0.04212129, ...,  0.00036081,
        -0.10750656,  0.03050067], dtype=float32),
 'foxi1': array([ 1.3280621e-02,  1.8214282e-02,  3.6576863e-02, ...,
         4.8775255e-05, -1.3445100e-01,  4.9920566e-02], dtype=float32),
 'hand2': array([ 0.05067997,  0.00322674,  0.03092083, ..., -0.00808235,
        -0.19103077,  0.03434367], dtype=float32),
 'hgfa': array([-0.01052155, -0.04286336, 

In [4]:
ood_conds

['zc4h2_control_24', 'met_control_36', 'tfap2a_control_72', 'hgfa_control_48', 'tfap2a_foxd3_72', ..., 'tbx16_msgn1_18', 'epha4a_control_18', 'tbxta_control_18', 'hoxb1a_control_36', 'hoxb1a_control_24']
Length: 71
Categories (71, object): ['cdx4_cdx1a_18', 'cdx4_cdx1a_24', 'cdx4_cdx1a_36', 'cdx4_control_18', ..., 'wnt3a_wnt8_36', 'zc4h2_control_24', 'zc4h2_control_36', 'zc4h2_control_48']

In [11]:
row.iloc[0]

cell                              A03_B01_P01-A01_LIG120
Size_Factor                                     0.757266
n.umi                                              344.0
perc_mitochondrial_umis                         2.906977
timepoint                                           24.0
Oligo                                  24h_zc4h2_P10_A10
hash_umis                                           15.0
top_to_second_best_ratio                        8.023582
cell_type_sub               pharyngeal arch (NC-derived)
cell_type_broad             pharyngeal arch (NC-derived)
tissue                                   Pharyngeal Arch
germ_layer                                  neural crest
log.n.umi                                       2.536558
num_genes_expressed                                  297
umap3d_1                                       -7.643328
umap3d_2                                       -3.773276
umap3d_3                                        0.501011
all_clust                      

In [1]:
import scanpy as sc


/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [2]:
import scanpy as sc
import functools
import os
import sys
import traceback
from typing import Dict, Literal, Optional, Tuple
import cellflow
import scanpy as sc
import numpy as np
import functools
from ott.solvers import utils as solver_utils
import optax
from omegaconf import OmegaConf
from typing import NamedTuple, Any
import hydra
import wandb
import anndata as ad
import pandas as pd
import rapids_singlecell as rsc
import anndata as ad
import os
from cellflow.preprocessing import transfer_labels, compute_wknn, centered_pca, project_pca
from cellflow.metrics._metrics import compute_r_squared, compute_e_distance, compute_scalar_mmd, compute_sinkhorn_div


In [3]:


def compute_metrics(adata_ref: ad.AnnData, adata_ref_for_ct_error: ad.AnnData, adata_pred: ad.AnnData, adata_ood_true: ad.AnnData, n_neighbors: int=1, cell_type_col: str = "cell_type_broad", min_cells_for_dist_metrics: int = 50) -> dict:
    dict_to_log = {}
    ct_true = adata_ood_true.obs[cell_type_col].value_counts().to_frame()
    ct_true = ct_true / ct_true.sum()
    
    compute_wknn(ref_adata=adata_ref_for_ct_error, query_adata=adata_pred, n_neighbors=n_neighbors, ref_rep_key="X_aligned", query_rep_key="X_aligned")
    transfer_labels(query_adata=adata_pred, ref_adata=adata_ref_for_ct_error, label_key=cell_type_col)
    ct_transferred_pred = adata_pred.obs[f"{cell_type_col}_transfer"].value_counts().to_frame()
    ct_transferred_pred/=ct_transferred_pred.sum()
    all_cell_types = list(set(ct_true.index).union(set(ct_transferred_pred.index)))
    df_all_cell_types = pd.DataFrame(index=all_cell_types, data=np.zeros((len(all_cell_types), 2)), columns=["true", "pred"])
    df_all_cell_types["true"] = ct_true
    df_all_cell_types["pred"] = ct_transferred_pred
    df_all_cell_types = df_all_cell_types.fillna(0.0)
    cell_type_fraction_error = np.abs(df_all_cell_types["true"] - df_all_cell_types["pred"]).sum()
    
    all_cell_types_true = list(adata_ood_true.obs[cell_type_col].value_counts()[adata_ood_true.obs[cell_type_col].value_counts()>min_cells_for_dist_metrics].index)
    compute_wknn(ref_adata=adata_ref, query_adata=adata_pred, n_neighbors=n_neighbors, ref_rep_key="X_aligned", query_rep_key="X_aligned")
    transfer_labels(query_adata=adata_pred, ref_adata=adata_ref, label_key=cell_type_col)
    
    e_distance = {}
    r_sq = {}
    mmd = {}
    sdiv_10 = {}
    sdiv_100 = {}
    n_cell_types_covered = 0
    for cell_type in all_cell_types_true: 
        dist_true = adata_ood_true[adata_ood_true.obs[f"{cell_type_col}"]==cell_type].obsm["X_aligned"]
        dist_pred = adata_pred[adata_pred.obs[f"{cell_type_col}_transfer"]==cell_type].obsm["X_aligned"]
        if len(dist_pred) == 0:
            continue
        n_cell_types_covered+=1
        r_sq[f"r_squared_{cell_type}"] = compute_r_squared(dist_true, dist_pred)
        e_distance[f"e_distance_{cell_type}"] = compute_e_distance(dist_true, dist_pred)
        mmd[f"mmd_{cell_type}"] = compute_scalar_mmd(dist_true, dist_pred)
        sdiv_10[f"div_10_{cell_type}"] = compute_sinkhorn_div(dist_true, dist_pred, epsilon=10.0)
        sdiv_100[f"div_100_{cell_type}"] = compute_sinkhorn_div(dist_true, dist_pred, epsilon=100.0)

    fraction_cell_types_covered = n_cell_types_covered/len(all_cell_types_true)


    # standard metrics
    ood_r_squared = compute_r_squared(adata_ood_true.obsm["X_aligned"], adata_pred.obsm["X_aligned"])
    ood_e_distance = compute_e_distance(adata_ood_true.obsm["X_aligned"], adata_pred.obsm["X_aligned"])
    ood_mmd = compute_scalar_mmd(adata_ood_true.obsm["X_aligned"], adata_pred.obsm["X_aligned"])
    ood_sdiv_10 = compute_sinkhorn_div(adata_ood_true.obsm["X_aligned"], adata_pred.obsm["X_aligned"], epsilon=10.0)
    ood_sdiv_100 = compute_sinkhorn_div(adata_ood_true.obsm["X_aligned"], adata_pred.obsm["X_aligned"], epsilon=100.0)
    
    # metrics to return
    dict_to_log["fraction_cell_types_covered"] = fraction_cell_types_covered
    dict_to_log["cell_type_fraction_error"] = cell_type_fraction_error
    dict_to_log["mean_r_sq_per_cell_type"] = np.mean(list(r_sq.values()))
    dict_to_log["mean_e_distance_per_cell_type"] = np.mean(list(e_distance.values()))
    dict_to_log["mean_mmd_per_cell_type"] = np.mean(list(mmd.values()))
    dict_to_log["mean_sdiv_10_per_cell_type"] = np.mean(list(sdiv_10.values()))
    dict_to_log["mean_sdiv_100_per_cell_type"] = np.mean(list(sdiv_100.values()))
    dict_to_log["median_r_sq_per_cell_type"] = np.median(list(r_sq.values()))
    dict_to_log["median_e_distance_per_cell_type"] = np.median(list(e_distance.values()))
    dict_to_log["median_mmd_per_cell_type"] = np.median(list(mmd.values()))
    dict_to_log["median_sdiv_10_per_cell_type"] = np.median(list(sdiv_10.values()))
    dict_to_log["median_sdiv_100_per_cell_type"] = np.median(list(sdiv_100.values()))
    dict_to_log.update(r_sq)
    dict_to_log.update(e_distance)
    dict_to_log.update(mmd)
    dict_to_log.update(sdiv_10)
    dict_to_log.update(sdiv_100)
    dict_to_log["ood_r_squared"] = ood_r_squared
    dict_to_log["ood_e_distance"] = ood_e_distance
    dict_to_log["ood_mmd"] = ood_mmd
    dict_to_log["ood_sdiv_10"] = ood_sdiv_10
    dict_to_log["ood_sdiv_100"] = ood_sdiv_100
    return dict_to_log

In [10]:
def get_train_embeddings(adata_same_timepoint: ad.AnnData, embeddings: dict[str, np.ndarray]) -> dict[str, Any]:
    conds = adata_same_timepoint.obs.drop_duplicates(subset="condition")
    emb_vectors = {}
    for _,row in conds.iterrows():
        g_1 = row["gene_target_1"]
        g_2  = row["gene_target_2"]
        if g_1=="control" and g_2=="control":
            continue
        elif g_1 != "control" and g_2!= "control":
            emb_vectors[(g_1, g_2)] = (embeddings[g_1] + embeddings[g_2])/2.0
        elif g_1 != "control" and g_2=="control":
            emb_vectors[(g_1, g_2)] = embeddings[g_1]
        elif g_1=="control" and g_2!="control":
            emb_vectors[(g_1, g_2)] = embeddings[g_2]
    return emb_vectors

def find_closest_embedding(emb_0: np.ndarray, reference_embeddings: dict[str, np.ndarray]) -> Tuple[str, ...]:
    closest_emb = None
    closest_dist = np.inf
    for ref, ref_emb in reference_embeddings.items():
        dist = np.sum((emb_0-ref_emb)**2)
        if dist < closest_dist:
            closest_dist = dist
            closest_emb = ref
    return closest_emb


In [5]:
adata = sc.read_h5ad("/lustre/groups/ml01/workspace/ot_perturbation/data/zebrafish_new/zebrafish_processed.h5ad")
out_dir = "/lustre/groups/ml01/workspace/ot_perturbation/models/identity/zebrafish/single_condition_closest_embedding"
ood_conds = adata[adata.obs["gene_target"]!="control_control"].obs["condition"].unique()
ood_cond_results = {}





In [7]:
adata.uns.keys()

dict_keys(['gene_embeddings', 'hvg', 'log1p'])

In [8]:
embeddings = adata.uns["gene_embeddings"]

In [ ]:

for ood_cond in ood_conds:
    print(ood_cond)
    adata_ood_true = adata[adata.obs["condition"]==ood_cond]
    adata_ref_for_ct_error = adata[adata.obs["condition"]!=ood_cond]
    tp = int(ood_cond.split("_")[-1])
    gene_knockout = "_".join(ood_cond.split("_")[:-1])
    print(gene_knockout)
    adata_same_timepoint = adata[adata.obs["timepoint"]==tp]
    same_timepoint_embeddings = get_train_embeddings(adata_same_timepoint, embeddings)
    
    if gene_knockout == "control_control":
        continue
    elif "control" in gene_knockout:
        emb_0 = ood_cond.split("_")[0]
        closest_emb = find_closest_embedding(embeddings[emb_0], same_timepoint_embeddings)
    else:
        emb_0 = (embeddings[ood_cond.split("_")[0]] + embeddings[ood_cond.split("_")[1]])/2.0
        closest_emb = find_closest_embedding(emb_0, same_timepoint_embeddings)
    
    adata_ood_pred = adata[adata.obs["condition"]==f"{closest_emb[0]}_{closest_emb[1]}_{tp}"]
    print(adata_ood_pred.n_obs)
    
    if adata_ood_pred.n_obs > 30000:
        sc.pp.subsample(adata_ood_pred, n_obs=30000)
    if adata_ood_true.n_obs > 30000:
        sc.pp.subsample(adata_ood_true, n_obs=30000)

    ood_cond_results[ood_cond] = compute_metrics(adata_ref=adata, adata_ref_for_ct_error=adata_ref_for_ct_error, adata_pred=adata_ood_pred, adata_ood_true=adata_ood_true)

    pd.DataFrame.from_dict(ood_cond_results[ood_cond], columns=[ood_cond], orient="index").to_csv(os.path.join(out_dir, f"{ood_cond}_closest_embedding.csv"))
    break

zc4h2_control_24
zc4h2_control
23513


/ictstr01/home/icb/dominik.klein/git_repos/cell_flow_perturbation/src/cellflow/preprocessing/_wknn.py:88: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  ref_adata.uns[uns_key_added] = wknn
/ictstr01/home/icb/dominik.klein/git_repos/cell_flow_perturbation/src/cellflow/preprocessing/_wknn.py:144: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  query_adata.obs[f"{label_key}_transfer"] = scores.idxmax(1)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/ott/solvers/linear/sinkhorn.py:924: UserWarning: Explicitly requested dtype float64 requested in ones is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  errors = -jnp.ones((self.outer_iterations, len(self.norm_error)),
/

In [ ]:
12

In [14]:
adata.obs.head()


,cell,Size_Factor,n.umi,perc_mitochondrial_umis,timepoint,Oligo,hash_umis,top_to_second_best_ratio,cell_type_sub,cell_type_broad,...,hash_plate,log.hash_umis,gene1+gene2,gene_target_1,gene_target_2,condition,is_control,first_t_control,logtime,germ_layer_adapted
A03_B01_P01-A01_LIG104,A03_B01_P01-A01_LIG104,0.766072,348.0,4.022989,18.0,18h_ctrl-inj_P13_H1,29.0,25.803177,head mesenchyme/PA cartilage,head mesenchyme/PA cartilage,...,NA,NaN,control+negative,control,control,control_control_18,True,True,2.890372,mesoderm_neural_crest
A03_B01_P01-A02_LIG101,A03_B01_P01-A02_LIG101,1.001617,455.0,9.010989,72.0,72h_ctrl-inj_P10_A6,46.0,5.384193,intestine (mid),intestine,...,NA,NaN,control+negative,control,control,control_control_72,True,False,4.276666,endoderm
A03_B01_P01-A02_LIG127,A03_B01_P01-A02_LIG127,2.007637,912.0,1.644737,36.0,36h_ctrl-inj_P7_A8,15.0,9.082616,basal cell (early),basal cell,...,NA,NaN,control+negative,control,control,control_control_36,True,False,3.583519,ectoderm_neural_crest
A03_B01_P01-A02_LIG257,A03_B01_P01-A02_LIG257,0.468889,213.0,0.938967,18.0,18h_ctrl-noto_P3_D7,24.0,21.347128,neural progenitor (hindbrain R7/8),neural progenitor (hindbrain R7/8),...,NA,NaN,control+negative,control,control,control_control_18,True,True,2.890372,ectoderm_neural_crest
A03_B01_P01-A03_LIG201,A03_B01_P01-A03_LIG201,0.537131,244.0,0.409836,48.0,48h_ctrl-noto_P16_H9,10.0,6.059433,"neurons (differentiating, contains peripheral)","neurons (differentiating, contains peripheral)",...,NA,NaN,control+negative,control,control,control_control_48,True,False,3.871201,ectoderm_neural_crest


In [13]:
adata.obs["gene_knockout"].unique()

KeyError: 'gene_knockout'